# ForenSort - training on Colab
**Before running:** Runtime -> Change runtime type -> **T4 GPU**.
Upload `forensort_code.zip` and `forensort_data.zip` (from the project's `colab_upload/` folder) to **My Drive/ForenSort/** first.
Run the cells top to bottom. Results (checkpoint, logs) are saved straight to Drive, so they survive a disconnect.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU - change the runtime type to T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ForenSort'
!ls -lh {DRIVE}

In [ ]:
# unpack to Colab's fast local disk (about 2.4 GB; takes a few minutes the first time)
!mkdir -p /content/ForenSort
!cd /content/ForenSort && unzip -q -o {DRIVE}/forensort_code.zip && unzip -q -o {DRIVE}/forensort_data.zip
!ls /content/ForenSort /content/ForenSort/data/manifests
!find /content/ForenSort/data/ebts -name '*.lffs' | wc -l   # expect 10092

In [ ]:
!pip install -q pyyaml
%cd /content/ForenSort

## Quick check (1 minute): a short run on 128 images to make sure everything works on this machine

In [ ]:
!python src/train.py configs/baseline.yaml --set name=colab_check train.epochs=2 train.limit_train=128 train.limit_val=64 data.num_workers=2 out_dir=/content/check_out

## Real training
Only run this after the check above worked. It saves the best checkpoint to Drive whenever validation improves.
Expect roughly 1-2.5 min per epoch on a T4 (up to 25 epochs; stops early if validation stops improving).

In [ ]:
!python src/train.py configs/baseline.yaml --set data.num_workers=2 out_dir={DRIVE}/outputs

## Results so far

In [ ]:
import pandas as pd
log = pd.read_csv(f'{DRIVE}/outputs/metrics/baseline_train_log.csv'); display(log)
log.plot(x='epoch', y=['train_loss','val_loss'], title='Loss'); log.plot(x='epoch', y=['val_macro_f1','val_qwk'], title='Validation agreement')